# Task-wise comparison



In [1]:
import sys
from itertools import combinations
from pathlib import Path

analysis_dir = Path.cwd() if (Path.cwd() / "metrics").exists() else Path.cwd() / "analysis"
if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

import pandas as pd
from scipy.stats import binomtest

from metrics.conditions import GAIA_CONFIG, OFFICEBENCH_CONFIG, load_card

In [2]:
CONDITIONS = ["BL-Lower", "BL-Upper", "Blueprint", "Playbook", "Adaptive System"]
SPLITS = (1, 2, 3)


def mcnemar_p(runs, condition_a, condition_b, split, min_tasks=50):
    one = runs[runs["split"] == split] # separate by split
    a = one[one["condition"] == condition_a].set_index("task_key")["is_success"]
    b = one[one["condition"] == condition_b].set_index("task_key")["is_success"]


    common = a.index.intersection(b.index) # get common task
    a, b = a.loc[common], b.loc[common]

    a_only = int(((a == 1) & (b == 0)).sum())
    b_only = int(((a == 0) & (b == 1)).sum())
    discordant = a_only + b_only #look at disagreement
    p = 1.0 if discordant == 0 else binomtest(min(a_only, b_only), discordant, 0.5).pvalue
    return p, a_only, b_only, len(common), discordant


def comparison_table(runs, conditions=CONDITIONS):
    """Per-split McNemar p-values for every pair of conditions."""
    rows = []
    for first, second in combinations(conditions, 2):
        results = [mcnemar_p(runs, first, second, split) for split in SPLITS]
        p_values = [r[0] for r in results]
        deltas = [r[2] - r[1] for r in results]          # b_only - a_only
        paired = [r[3] for r in results]
        discordant = [r[4] for r in results]

        rows.append(
            {
                "Comparison": f"{first} vs {second}",
                **{f"p (split {s})": round(p, 4) for s, p in zip(SPLITS, p_values)},
                "splits p<0.05": sum(p < 0.05 for p in p_values),
                "delta (pairs)": deltas,
                "discordant": discordant,
                "n paired": paired,
            }
        )
    return pd.DataFrame(rows).set_index("Comparison")

In [3]:
for card in ["rich", "sparse"]:
    df = load_card(card, GAIA_CONFIG)
    print(f"=== GAIA {card} - exact McNemar within each split ===")
    display(comparison_table(df))

=== GAIA rich - exact McNemar within each split ===


,p (split 1),p (split 2),p (split 3),splits p<0.05,delta (pairs),discordant,n paired
Comparison,,,,,,,
BL-Lower vs BL-Upper,0.4049,0.4244,0.5034,0,"[5, 5, 4]","[23, 25, 20]","[100, 100, 100]"
BL-Lower vs Blueprint,1.0000,1.0000,0.6636,0,"[0, 0, 3]","[18, 26, 21]","[100, 100, 100]"
BL-Lower vs Playbook,0.5034,0.3323,0.7905,0,"[4, -5, 2]","[20, 17, 14]","[100, 100, 100]"
BL-Lower vs Adaptive System,0.5235,0.4421,0.1435,0,"[4, -5, 7]","[22, 27, 17]","[100, 100, 100]"
BL-Upper vs Blueprint,0.4244,0.4049,1.0000,0,"[-5, -5, -1]","[25, 23, 27]","[100, 100, 100]"
BL-Upper vs Playbook,1.0000,0.0525,0.8238,0,"[-1, -10, -2]","[27, 22, 20]","[100, 100, 100]"
BL-Upper vs Adaptive System,1.0000,0.0639,0.7011,0,"[-1, -10, 3]","[25, 24, 27]","[100, 100, 100]"
Blueprint vs Playbook,0.5034,0.4244,1.0000,0,"[4, -5, -1]","[20, 25, 25]","[100, 100, 100]"
Blueprint vs Adaptive System,0.5716,0.3593,0.5716,0,"[4, -5, 4]","[28, 19, 28]","[100, 100, 100]"


=== GAIA sparse - exact McNemar within each split ===


,p (split 1),p (split 2),p (split 3),splits p<0.05,delta (pairs),discordant,n paired
Comparison,,,,,,,
BL-Lower vs BL-Upper,0.5034,0.4244,1.0,0,"[4, 5, 0]","[20, 25, 18]","[100, 100, 100]"
BL-Lower vs Blueprint,0.8036,1.0000,1.0,0,"[-2, -1, 1]","[16, 21, 19]","[100, 100, 100]"
BL-Lower vs Playbook,1.0000,1.0000,1.0,0,"[1, 1, 1]","[19, 25, 23]","[100, 100, 100]"
BL-Lower vs Adaptive System,0.6476,0.5572,1.0,0,"[-3, 4, 1]","[19, 26, 25]","[100, 100, 100]"
BL-Upper vs Blueprint,0.2379,0.2863,1.0,0,"[-6, -6, 1]","[18, 22, 17]","[100, 100, 100]"
BL-Upper vs Playbook,0.6476,0.5413,1.0,0,"[-3, -4, 1]","[19, 24, 25]","[100, 100, 100]"
BL-Upper vs Adaptive System,0.1892,1.0000,1.0,0,"[-7, -1, 1]","[21, 29, 25]","[100, 100, 100]"
Blueprint vs Playbook,0.5078,0.8318,1.0,0,"[3, 2, 0]","[9, 22, 22]","[100, 100, 100]"
Blueprint vs Adaptive System,1.0000,0.3833,1.0,0,"[-1, 5, 0]","[13, 21, 24]","[100, 100, 100]"


In [4]:
for card in ["rich", "sparse"]:
    df = load_card(card, OFFICEBENCH_CONFIG)
    print(f"=== OFFICEBENCH {card} - exact McNemar within each split ===")
    display(comparison_table(df))

=== OFFICEBENCH rich - exact McNemar within each split ===


,p (split 1),p (split 2),p (split 3),splits p<0.05,delta (pairs),discordant,n paired
Comparison,,,,,,,
BL-Lower vs BL-Upper,0.0003,0.0019,0.0017,3,"[22, 19, 20]","[36, 35, 38]","[183, 173, 186]"
BL-Lower vs Blueprint,0.0000,0.0034,0.0001,3,"[38, 19, 27]","[46, 39, 47]","[183, 173, 186]"
BL-Lower vs Playbook,0.0001,0.1996,0.0357,2,"[28, 9, 15]","[48, 39, 45]","[183, 173, 186]"
BL-Lower vs Adaptive System,0.0000,0.0025,0.0066,3,"[37, 21, 20]","[57, 45, 50]","[183, 173, 186]"
BL-Upper vs Blueprint,0.0259,1.0000,0.3368,1,"[16, 0, 7]","[46, 42, 39]","[183, 173, 186]"
BL-Upper vs Playbook,0.4885,0.1742,0.5758,0,"[6, -10, -5]","[52, 44, 51]","[183, 173, 186]"
BL-Upper vs Adaptive System,0.0534,0.8804,1.0000,0,"[15, 2, 0]","[53, 44, 52]","[183, 173, 186]"
Blueprint vs Playbook,0.1934,0.1934,0.1114,0,"[-10, -10, -12]","[48, 48, 48]","[183, 173, 186]"
Blueprint vs Adaptive System,1.0000,0.8804,0.3489,0,"[-1, 2, -7]","[51, 44, 41]","[183, 173, 186]"


=== OFFICEBENCH sparse - exact McNemar within each split ===


,p (split 1),p (split 2),p (split 3),splits p<0.05,delta (pairs),discordant,n paired
Comparison,,,,,,,
BL-Lower vs BL-Upper,0.0192,0.1078,0.0146,2,"[13, 9, 13]","[27, 25, 25]","[183, 173, 186]"
BL-Lower vs Blueprint,0.0000,0.0029,0.0000,3,"[31, 18, 30]","[41, 34, 36]","[183, 173, 186]"
BL-Lower vs Playbook,0.0002,0.0002,0.0137,3,"[27, 24, 17]","[51, 40, 43]","[183, 173, 186]"
BL-Lower vs Adaptive System,0.0038,0.0000,0.0000,3,"[22, 31, 29]","[54, 57, 45]","[183, 173, 186]"
BL-Upper vs Blueprint,0.0051,0.1996,0.0060,2,"[18, 9, 17]","[38, 39, 35]","[183, 173, 186]"
BL-Upper vs Playbook,0.0436,0.0400,0.6587,2,"[14, 15, 4]","[42, 47, 46]","[183, 173, 186]"
BL-Upper vs Adaptive System,0.2221,0.0071,0.0259,2,"[9, 22, 16]","[43, 62, 46]","[183, 173, 186]"
Blueprint vs Playbook,0.6177,0.4614,0.0596,0,"[-4, 6, -13]","[36, 46, 41]","[183, 173, 186]"
Blueprint vs Adaptive System,0.2221,0.0854,1.0000,0,"[-9, 13, -1]","[43, 49, 35]","[183, 173, 186]"
